# Notebook 01 - Pre-Training-Daten (Synthetisch, auf ECHTER Geometrie)
### Phase 1: Vortrainingsdaten fuer LSTM/TFT

Erzeugt synthetische Zeitreihen-WERTE, aber auf der ECHTEN Planungsraum-
Geometrie (542 PLR, LOR-Stand 2021) - dieselbe Struktur wie die spaeteren
empirischen Daten. Nur die Attributwerte selbst sind simuliert, nicht die
raeumliche Topologie.

**Feature-Set** (muss exakt zu Notebook 02/03/04 passen!):
`arbeitslosigkeit, sgb2_quote, kinderarmut, dist_zentrum, status_lag`

**Voraussetzung:** Notebook 02, Zellen 1-3 sollten vorher einmal gelaufen
sein (Geometrie-Cache). Falls nicht, laedt Zelle 2 das Shapefile selbst.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 1: Imports & Setup
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import os
from shapely.geometry import Point

np.random.seed(42)

for folder in ['../data/raw', '../data/processed', '../data/final',
               '../models', '../results']:
    os.makedirs(folder, exist_ok=True)

print('✅ Setup abgeschlossen!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 2: Echte Planungsraum-Geometrie laden (542 PLR, LOR 2021)
# ═══════════════════════════════════════════════════════════════

cache_path = '../data/processed/plr_geometrie_cache.csv'

if os.path.exists(cache_path):
    geo = pd.read_csv(cache_path)
    geo['plr_id'] = geo['plr_id'].astype(str).str.zfill(8)
    print(f'✅ Geometrie aus Cache geladen: {len(geo)} PLR')
else:
    print('⚠️ Kein Cache gefunden - lade Shapefile direkt...')
    gdf_neu = gpd.read_file('../data/raw/planungsraeume_neu/LOR_2023-01-01_PLR_EPSG_25833_nur_ID.shp')
    gdf_neu = gdf_neu.to_crs(epsg=25833)

    id_col = next(
        (c for c in gdf_neu.columns if c.upper() in ('PLR_ID', 'PLR_ID_STR', 'SCHLUESSEL')),
        gdf_neu.columns[0]
    )
    gdf_neu = gdf_neu.rename(columns={id_col: 'plr_id'})
    gdf_neu['plr_id'] = gdf_neu['plr_id'].astype(str).str.strip().str.zfill(8)

    gdf_neu['centroid_x'] = gdf_neu.geometry.centroid.x
    gdf_neu['centroid_y'] = gdf_neu.geometry.centroid.y
    alex = Point(392456, 5820118)
    gdf_neu['dist_zentrum'] = (gdf_neu.geometry.centroid.distance(alex) / 1000).round(2)

    geo = gdf_neu[['plr_id', 'centroid_x', 'centroid_y', 'dist_zentrum']].copy()
    geo.to_csv(cache_path, index=False)
    print(f'✅ Shapefile geladen & gecacht: {len(geo)} PLR')

plr_ids      = geo['plr_id'].tolist()
N_PLR        = len(plr_ids)
centroid_x   = geo['centroid_x'].values
centroid_y   = geo['centroid_y'].values
dist_zentrum = geo['dist_zentrum'].values

print(f'\n✅ Basis fuer synthetische Daten: {N_PLR} ECHTE Planungsraeume')
print(f'   (erwartet: 542, LOR-Stand 2021)')

if N_PLR < 100:
    print('\n⚠️  WARNUNG: Sehr wenige PLR geladen - Shapefile-Pfad/Spaltenname pruefen!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 3: Synthetische Zeitreihen-WERTE generieren
# ═══════════════════════════════════════════════════════════════
# Geometrie ist ECHT (aus Zelle 2), nur die Attributwerte sind simuliert.
# Feature-Set entspricht bewusst Notebook 02 (ohne oepnv/gruen/laerm,
# ohne status_index als Feature - siehe dortige Begruendung).

JAHRE = list(range(2000, 2024, 1))

bezirke = [
    'Mitte', 'Neukoelln', 'Kreuzberg', 'Prenzlauer Berg',
    'Marzahn', 'Wedding', 'Charlottenburg', 'Spandau', 'Tempelhof'
]

plr_bezirk   = np.random.choice(bezirke, N_PLR)
basis_status = np.random.normal(0, 1.2, N_PLR)

rows = []
stadttyp_liste = []

for i, plr_id in enumerate(plr_ids):
    status = basis_status[i]

    stadttyp = np.random.choice(
        ['gentrifizierend', 'stabil', 'abwertend', 'volatil'],
        p=[0.25, 0.40, 0.20, 0.15]
    )
    stadttyp_liste.append(stadttyp)

    if stadttyp == 'gentrifizierend':
        trend = np.random.uniform(0.03, 0.10)
    elif stadttyp == 'abwertend':
        trend = np.random.uniform(-0.10, -0.03)
    elif stadttyp == 'volatil':
        trend = 0.0
    else:
        trend = np.random.uniform(-0.01, 0.01)

    noise_scale = 0.08 if stadttyp == 'volatil' else 0.03

    for year in JAHRE:
        status += trend + np.random.normal(0, noise_scale)
        noise = np.random.normal(0, 0.02)

        rows.append({
            'plr_id':      plr_id,
            'bezirk':      plr_bezirk[i],
            'stadttyp':    stadttyp,
            'year':        year,
            'status_index': round(status, 3),
            'arbeitslosigkeit': round(max(0, 0.10 - status*0.025 + noise), 4),
            'sgb2_quote':       round(max(0, 0.18 - status*0.035 + noise), 4),
            'kinderarmut':      round(max(0, 0.22 - status*0.045 + noise), 4),
            'centroid_x':   round(float(centroid_x[i]), 2),
            'centroid_y':   round(float(centroid_y[i]), 2),
            'dist_zentrum': round(float(dist_zentrum[i]), 2),
        })

df_synth = pd.DataFrame(rows)

print(f'✅ Rohdaten generiert: {df_synth.shape}')
print(f'   PLR:   {df_synth["plr_id"].nunique()} (= echte Berliner Planungsraeume)')
print(f'   Jahre: {df_synth["year"].min()}-{df_synth["year"].max()}')
print(f'\nStadttyp-Verteilung:')
print(pd.Series(stadttyp_liste).value_counts())

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 4: Raeumlicher Lag (Proxy) & Ziel-Variable
# ═══════════════════════════════════════════════════════════════
# status_lag als einfacher Jahresmittelwert-Proxy (fuer Pre-Training
# ausreichend - echte Queen-Nachbarschaft erst in Notebook 02).
# WICHTIG: muss hier vorhanden sein, damit FEATURE_COLS_COMMON in
# Notebook 03/04 'status_lag' nicht faelschlich ausschliesst.

df_synth = df_synth.sort_values(['plr_id', 'year'])

df_synth['status_lag'] = df_synth.groupby('year')['status_index'].transform('mean')

df_synth['status_next'] = df_synth.groupby('plr_id')['status_index'].shift(-1)
df_synth['delta_status'] = df_synth['status_next'] - df_synth['status_index']
df_synth = df_synth.dropna(subset=['status_next', 'delta_status'])

year_map = {y: i for i, y in enumerate(sorted(df_synth['year'].unique()))}
df_synth['time_idx'] = df_synth['year'].map(year_map)

print(f'✅ Ziel-Variablen berechnet: {df_synth.shape}')
print(df_synth[['status_index','status_lag','status_next','delta_status']].describe().round(3))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 5: Visualisierung & Speichern
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for typ, color in zip(['gentrifizierend','stabil','abwertend','volatil'],
                       ['green','steelblue','red','orange']):
    sample_plr = df_synth[df_synth['stadttyp']==typ]['plr_id'].unique()[:5]
    for plr in sample_plr:
        sub = df_synth[df_synth['plr_id']==plr].sort_values('year')
        axes[0].plot(sub['year'], sub['status_index'], color=color, alpha=0.6,
                    label=typ if plr==sample_plr[0] else None)
axes[0].set_title('Beispielzeitreihen je Stadttyp')
axes[0].set_xlabel('Jahr'); axes[0].set_ylabel('Status-Index')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].hist(df_synth['status_index'], bins=40, color='steelblue',
             edgecolor='white', alpha=0.8)
axes[1].set_title('Verteilung Status-Index (gesamt)')
axes[1].set_xlabel('status_index')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Pre-Training-Datensatz: {N_PLR} echte PLR x {len(JAHRE)} Jahre',
             fontweight='bold')
plt.tight_layout()
plt.savefig('../results/01_pretraining_uebersicht.png', dpi=150)
plt.show()

df_synth.to_csv('../data/raw/mss_pretraining_synthetisch.csv', index=False)
print('✅ Gespeichert: data/raw/mss_pretraining_synthetisch.csv')
print(f'   Shape: {df_synth.shape}')
print(f'   Features: arbeitslosigkeit, sgb2_quote, kinderarmut, dist_zentrum, status_lag')
print('\n→ Weiter mit Notebook 03 (LSTM Pre-Training + Fine-Tuning)')